# Coleta Dados Deputado


In [ ]:
import requests
import json
import time
import csv

# --- CONFIGURAÇÕES ---
ANOS_INTERESSE = [2022, 2023, 2024, 2025, 2026]
LEGISLATURA_ATUAL = 57
HEADERS = {'Accept': 'application/json'}

# --- PASSO 1: LISTA DE DEPUTADOS ---
url_deputados = "https://dadosabertos.camara.leg.br/api/v2/deputados"
params_dep = {'idLegislatura': LEGISLATURA_ATUAL, 'ordem': 'ASC', 'ordenarPor': 'nome'}

print("--- Passo 1: Coletando lista de deputados ---")
res = requests.get(url_deputados, params=params_dep)
lista_deputados = res.json()['dados']

# Listas para armazenar os dados filtrados
extrato_despesas = []
extrato_presencas = []
extrato_projetos = []

IDS_APROVADOS = [114, 924]

# --- PASSO 2: LOOP DE COLETA ---
for deputado in lista_deputados:
    d_id = deputado['id']
    d_nome = deputado['nome']
    d_uf = deputado['siglaUf']

    print(f"Processando: {d_nome} ({d_uf})...")

    # 1. DESPESAS
    for ano in ANOS_INTERESSE:
        try:
            url_g = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{d_id}/despesas"
            res_g = requests.get(url_g, params={'ano': ano}, timeout=10)
            if res_g.status_code == 200:
                for g in res_g.json()['dados']:
                    extrato_despesas.append({
                        'nome': d_nome, 'uf': d_uf, 'ano': ano,
                        'data': g.get('dataEmissao'),
                        'tipo': g.get('tipoDespesa'),
                        'valor': g.get('valorLiquido')
                    })
        except: pass

    # 2. PRESENÇAS
    try:
        url_e = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{d_id}/eventos"
        params_e = {'dataInicio': '2022-01-01', 'dataFim': '2026-12-31'}
        res_e = requests.get(url_e, params=params_e, timeout=10)
        if res_e.status_code == 200:
            for e in res_e.json()['dados']:
                extrato_presencas.append({
                    'nome': d_nome, 'uf': d_uf,
                    'data': e.get('dataHoraInicio'),
                    'tipo_evento': e.get('descricaoTipo'),
                    'situacao': e.get('situacao')
                })
    except: pass

    # 3. PROJETOS (Corrigido para não vir vazio)
    try:
        url_p = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
        for ano_p in ANOS_INTERESSE:
            res_p = requests.get(url_p, params={'idDeputadoAutor': d_id, 'ano': ano_p}, timeout=10)
            if res_p.status_code == 200:
                for p in res_p.json()['dados']:
                    extrato_projetos.append({
                        'nome': d_nome, 'uf': d_uf, 'ano': ano_p,
                        'tipo_projeto': p.get('siglaTipo'),
                        'ementa': p.get('ementa'),
                        'foi_aprovado': p.get('ultimoStatus', {}).get('idSituacao') in IDS_APROVADOS
                    })
    except: pass

    time.sleep(0.4) # Respiro para a API não te bloquear

# --- PASSO 3: SALVAMENTO EM CSV (O SEGREDO DA LEVEZA) ---
print("\n--- Passo 3: Gerando arquivos CSV para o time ---")

def salvar_csv(nome_arquivo, dados, colunas):
    with open(nome_arquivo, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=colunas, delimiter=';')
        writer.writeheader()
        writer.writerows(dados)

# Salvando cada categoria
if extrato_despesas:
    salvar_csv("radar_despesas.csv", extrato_despesas, ['nome', 'uf', 'ano', 'data', 'tipo', 'valor'])

if extrato_presencas:
    salvar_csv("radar_presencas.csv", extrato_presencas, ['nome', 'uf', 'data', 'tipo_evento', 'situacao'])

if extrato_projetos:
    salvar_csv("radar_projetos.csv", extrato_projetos, ['nome', 'uf', 'ano', 'tipo_projeto', 'ementa', 'foi_aprovado'])

print("\n--- TUDO PRONTO! ---")
print(f"Arquivos CSV gerados com sucesso. Agora seu time pode abrir no Excel!")

# Coleta Deputados ID

In [ ]:
import requests
import csv
import time

# --- CONFIGURAÇÕES ---
LEGISLATURA = 57
HEADERS = {'Accept': 'application/json'} # Forçamos JSON para ser mais fácil que XML

# 1. Pegar a lista básica para saber quem são os 513 deputados
print(f"Buscando lista de IDs da Legislatura {LEGISLATURA}...")
url_lista = f"https://dadosabertos.camara.leg.br/api/v2/deputados?idLegislatura={LEGISLATURA}"
res_lista = requests.get(url_lista, headers=HEADERS)
deputados_basico = res_lista.json()['dados']

detalhes_finais = []

# 2. Percorrer cada deputado para buscar o CPF e a Foto no endpoint específico
print(f"Iniciando coleta detalhada de {len(deputados_basico)} deputados. Isso vai levar uns 4 minutos...")

for dep in deputados_basico:
    dep_id = dep['id']
    url_detalhe = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{dep_id}"

    try:
        res_detalhe = requests.get(url_detalhe, headers=HEADERS, timeout=10)
        if res_detalhe.status_code == 200:
            info = res_detalhe.json()['dados']

            # Verificação de segurança: Só pegamos se ele realmente estiver na legislatura 57
            if info.get('ultimoStatus', {}).get('idLegislatura') == LEGISLATURA:
                detalhes_finais.append({
                    'id': dep_id,
                    'nome': info.get('nomeCivil'),
                    'nome_eleitoral': info.get('ultimoStatus', {}).get('nome'),
                    'cpf': info.get('cpf'),
                    'urlFoto': info.get('ultimoStatus', {}).get('urlFoto'),
                    'partido': info.get('ultimoStatus', {}).get('siglaPartido'),
                    'uf': info.get('ultimoStatus', {}).get('siglaUf'),
                    'situacao': info.get('ultimoStatus', {}).get('situacao')
                })
                print(f"OK: {info.get('ultimoStatus', {}).get('nome')}")

        # Pausa curta para não sobrecarregar a API da Câmara
        time.sleep(0.2)

    except Exception as e:
        print(f"Erro ao buscar detalhes do ID {dep_id}: {e}")

# 3. Salvar o resultado em um CSV limpo
colunas = ['id', 'nome', 'nome_eleitoral', 'cpf', 'urlFoto', 'partido', 'uf', 'situacao']

with open('deputados_detalhes_com_foto.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=colunas, delimiter=';')
    writer.writeheader()
    writer.writerows(detalhes_finais)

print("\n--- PROCESSO CONCLUÍDO ---")
print(f"Arquivo 'deputados_detalhes_com_foto.csv' gerado com {len(detalhes_finais)} registros.")

# Coleta Dados Aprovados


In [ ]:
import requests
import csv
import time

# --- CONFIGURAÇÕES ---
HEADERS = {'Accept': 'application/json'}
URL_BASE = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"

# Vamos definir os trimestres para contornar a regra dos 3 meses
# Formato: (Data Inicio, Data Fim)
PERIODOS = [

    ('2003-01-01', '2003-03-31'),
    ('2003-04-01', '2003-06-30'),
    ('2003-07-01', '2003-09-30'),
    ('2003-10-01', '2003-12-31'),
    ('2004-01-01', '2004-03-31'),
    ('2004-04-01', '2004-06-30'),
    ('2004-07-01', '2004-09-30'),
    ('2004-10-01', '2004-12-31'),
    ('2005-01-01', '2005-03-31'),
    ('2005-04-01', '2005-06-30'),
    ('2005-07-01', '2005-09-30'),
    ('2005-10-01', '2005-12-31'),
    ('2006-01-01', '2006-03-31'),
    ('2006-04-01', '2006-06-30'),
    ('2006-07-01', '2006-09-30'),
    ('2006-10-01', '2006-12-31'),
    ('2007-01-01', '2007-03-31'),
    ('2007-04-01', '2007-06-30'),
    ('2007-07-01', '2007-09-30'),
    ('2007-10-01', '2007-12-31'),
    ('2008-01-01', '2008-03-31'),
    ('2008-04-01', '2008-06-30'),
    ('2008-07-01', '2008-09-30'),
    ('2008-10-01', '2008-12-31'),
    ('2009-01-01', '2009-03-31'),
    ('2009-04-01', '2009-06-30'),
    ('2009-07-01', '2009-09-30'),
    ('2009-10-01', '2009-12-31'),
    ('2010-01-01', '2010-03-31'),
    ('2010-04-01', '2010-06-30'),
    ('2010-07-01', '2010-09-30'),
    ('2010-10-01', '2010-12-31'),
    ('2011-01-01', '2011-03-31'),
    ('2011-04-01', '2011-06-30'),
    ('2011-07-01', '2011-09-30'),
    ('2011-10-01', '2011-12-31'),
    ('2012-01-01', '2012-03-31'),
    ('2012-04-01', '2012-06-30'),
    ('2012-07-01', '2012-09-30'),
    ('2012-10-01', '2012-12-31'),
    ('2013-01-01', '2013-03-31'),
    ('2013-04-01', '2013-06-30'),
    ('2013-07-01', '2013-09-30'),
    ('2013-10-01', '2013-12-31'),
    ('2014-01-01', '2014-03-31'),
    ('2014-04-01', '2014-06-30'),
    ('2014-07-01', '2014-09-30'),
    ('2014-10-01', '2014-12-31'),
    ('2015-01-01', '2015-03-31'),
    ('2015-04-01', '2015-06-30'),
    ('2015-07-01', '2015-09-30'),
    ('2015-10-01', '2015-12-31'),
    ('2016-01-01', '2016-03-31'),
    ('2016-04-01', '2016-06-30'),
    ('2016-07-01', '2016-09-30'),
    ('2016-10-01', '2016-12-31'),
    ('2017-01-01', '2017-03-31'),
    ('2017-04-01', '2017-06-30'),
    ('2017-07-01', '2017-09-30'),
    ('2017-10-01', '2017-12-31'),
    ('2018-01-01', '2018-03-31'),
    ('2018-04-01', '2018-06-30'),
    ('2018-07-01', '2018-09-30'),
    ('2018-10-01', '2018-12-31'),
    ('2019-01-01', '2019-03-31'),
    ('2019-04-01', '2019-06-30'),
    ('2019-07-01', '2019-09-30'),
    ('2019-10-01', '2019-12-31'),
    ('2020-01-01', '2020-03-31'),
    ('2020-04-01', '2020-06-30'),
    ('2020-07-01', '2020-09-30'),
    ('2020-10-01', '2020-12-31'),
    ('2021-01-01', '2021-03-31'),
    ('2021-04-01', '2021-06-30'),
    ('2021-07-01', '2021-09-30'),
    ('2021-10-01', '2021-12-31'),
]

leis_aprovadas = []
contagem_por_deputado = {}

print(f"--- Iniciando Busca Fracionada (Respeitando limite de 3 meses) ---")

for inicio, fim in PERIODOS:
    print(f"Buscando período: {inicio} até {fim}...")

    params = {
        'siglaTipo': 'PL',
        'dataInicio': inicio,
        'dataFim': fim,
        'codSituacao': 1140, # Transformada em Norma Jurídica
        'itens': 100,
        'ordem': 'ASC',
        'ordenarPor': 'id'
    }

    try:
        res = requests.get(URL_BASE, params=params, headers=HEADERS, timeout=30)

        if res.status_code == 200:
            dados = res.json().get('dados', [])
            print(f"  > Encontrados {len(dados)} projetos neste período.")

            for p in dados:
                id_prop = p['id']
                # Pequena pausa para não ser bloqueado pela API (Rate Limit)
                time.sleep(0.1)

                res_autores = requests.get(f"{URL_BASE}/{id_prop}/autores", headers=HEADERS)

                nomes_autores = []
                if res_autores.status_code == 200:
                    autores_lista = res_autores.json().get('dados', [])
                    for autor in autores_lista:
                        nome = autor['nome']
                        nomes_autores.append(nome)
                        contagem_por_deputado[nome] = contagem_por_deputado.get(nome, 0) + 1

                leis_aprovadas.append({
                    'Autor Principal': nomes_autores[0] if nomes_autores else "Desconhecido",
                    'Tipo': p['siglaTipo'],
                    'Numero': p['numero'],
                    'Ano': p['ano'],
                    'Ementa': p['ementa'],
                    'Periodo_Busca': f"{inicio} a {fim}"
                })
        else:
            print(f"  > Erro {res.status_code} no período {inicio}: {res.text}")

    except Exception as e:
        print(f"  > Erro técnico no período {inicio}: {e}")

# --- SALVAMENTO ---
if leis_aprovadas:
    # 1. Ranking
    with open('ranking_vitoriosos.csv', 'w', newline='', encoding='utf-8-sig') as f1:
        writer = csv.writer(f1, delimiter=';')
        writer.writerow(['Deputado', 'Qtd Leis Aprovadas'])
        for nome, total in sorted(contagem_por_deputado.items(), key=lambda x: x[1], reverse=True):
            writer.writerow([nome, total])

    # 2. Detalhes
    with open('detalhes_leis_aprovadas.csv', 'w', newline='', encoding='utf-8-sig') as f2:
        writer = csv.DictWriter(f2, fieldnames=leis_aprovadas[0].keys(), delimiter=';')
        writer.writeheader()
        writer.writerows(leis_aprovadas)

    print(f"\n--- SUCESSO! ---")
    print(f"Total de leis encontradas: {len(leis_aprovadas)}")
    print("Arquivos 'ranking_vitoriosos.csv' e 'detalhes_leis_aprovadas.csv' gerados.")
else:
    print("\nA busca não retornou resultados. Tente mudar o 'codSituacao' para 921 (Sanção).")